# MAKRO 2026: a simple experiment with standard ModelFlow

Public purchases of goods and services up by 1 per cent, permanently from 2030. That is about 0.1 per cent of
GDP, small enough to solve in one step.

Only standard ModelFlow is used: `modelload`, the model call with `keep=`, `.upd`, `fix`, and the standard
tables and charts. `makro26_experiment.ipynb` does more (DREAM's standard shocks, the two-step solve for
large shocks, comparison with DREAM).

**What it needs:** ModelFlow from `C:\modelflow2\modelflow` (the `factorizer=` option, `modelfactorize` and the
solve options of November 2026: `newton_stall_refresh`, `ng_solve_status`),
`pypardiso` with oneMKL 2026 or newer, numba, and about 25 GB of free memory. The baseline solve takes about
a minute (the factorization), the experiment about 20 seconds. The very first run also compiles the equations
(about 20 minutes, saved for later runs).

In [1]:
from modelclass import model
model.plain_html = True   # results as plain html tabs, so they also show in the book

## 1. Load

The file holds the equations, the baseline 2015-2131, the descriptions, the variable groups, the derivative
expressions and the solve options. It was saved from a model loaded with `straight=True` (skip the analysis of
the simultaneous structure, 2 minutes, which the stacked Newton does not use), and the load does the same. So
the file name is all the load needs.

In [2]:
%%time
mmakro, baseline = model.modelload(r'C:\mfmodeller_raw\makro\makro26\data\makro26_baseline.pcimz')
print('Solve options saved with the model:', mmakro.oldkwargs)

Feather/zip file read: C:\mfmodeller_raw\makro\makro26\data\makro26_baseline.pcimz


Model: MAKRO 2026-September (DREAM) in ModelFlow, baseline
Solve options saved with the model: {'solver': 'newtonstack_ng', 'forcenum': False, 'factorizer': {'method': 'normal', 'iparm': {'1': 13}, 'refine_tol': 1e-06}, 'silent': 1}
CPU times: total: 1.58 s
Wall time: 1.2 s


## 2. Run the baseline

MAKRO looks forward, so all years are solved together. The span must end in the terminal year 2129.

The solve options come with the file; a model call uses them unless the call says otherwise:

| Saved option | Why |
| --- | --- |
| `solver='newtonstack_ng'` | all years in one stacked Newton system |
| `forcenum=False` | exact derivatives; numerical ones are not accurate enough for MAKRO |
| `factorizer={'method': 'normal', 'iparm': {1: 13}, 'refine_tol': 1e-6}` | the linear system (1.5 million unknowns) through the normal equations and PARDISO, see below |

The call adds:

| Option | Why |
| --- | --- |
| `ljit=True` | the equations compiled with numba, see below |
| `factorizer=fast` | the saved factorizer with the linear solves refined only to 5e-3, see below |
| `keep='Baseline'` | keep the solution for the charts |
| `setbase=True` | the solution becomes the base that later results are compared with |

The factorizer is the method `'normal'` with two settings. `iparm={1: 13}` is the faster factorization of
oneMKL 2026 (with an older MKL under `pypardiso` ModelFlow leaves it out with a note). `refine_tol` stops the
refinement of each linear solve at that relative residual instead of 1e-11. A solve with an existing
factorization spends most of its time in these refinements, and Newton corrects an inexact step in its next
iteration. The model file has 1e-6 (the experiment solve 50 → 27 s); `fast` uses 5e-3, which as a rule is one
application of the factors (tested on all of DREAM's standard shocks: the same solutions to the convergence
tolerance, a few more iterations; public purchases +1 per cent of GDP 48 → 33 s). ModelFlow refuses 1e-2 and above. Written as a dict,
the choice is saved with the model (`modeldump`), so a model loaded from the file solves the same way.

`ljit=True` compiles the equations with numba: one Newton iteration evaluates them in 0.01 s instead of 2 s.
The first time it takes about 20 minutes. The compiled code is saved in `modelsource\` in the working
directory and loaded from there in later sessions; ModelFlow compiles again only when the equations change.
Because the compiled code belongs to the working directory, `ljit` is not saved with the model: a program
started elsewhere would otherwise compile for 20 minutes before its first solve.

ModelFlow keeps the factorization between solves as long as the factorizer option is the same, so the
experiment below does not factorize again: it uses the derivatives of the baseline. ModelFlow also remembers the options of the last call, so the
experiment uses `ljit=True` too. The baseline data already solves the model, so the run should take one
iteration and change nothing.

In [3]:
%%time
first, last = 2030, 2129
fast = {**mmakro.oldkwargs['factorizer'], 'refine_tol': 5e-3}   # the saved factorizer, faster linear solves
base = mmakro(baseline, first, last, ljit=True, factorizer=fast, keep='Baseline', setbase=True)

CPU times: total: 7min 21s
Wall time: 58.8 s


## 3. The experiment

`QR__OFF` is public purchases of goods and services, `qR['off',t]` in MAKRO. The update line raises it by
1 per cent in all years.

In [4]:
mmakro['QR__OFF'].des

QR__OFF : Materialeinput fordelt på brancherne som modtager materialeinputtet, Kilde: ADAM[fVm] eller ADAM[fVm<i>] [OFF] (MAKRO qR)


In [5]:
alt = base.upd(f'<{first} {last}> QR__OFF * 1.01')

**The first year.** A shock is unanticipated. GAMS leaves a few equations out in the first year of a run: the
required return on shares, the firms' discount rate and factor and their risk premia (39 variables). Their
values for that year were set before the shock. `fix` does the same here: it fixes these variables at their
baseline value in 2030 (sets `_D` = 1 and `_X` = the value).

In [6]:
alt = mmakro.fix(alt, 'RAKTIEDRIFT* RVIRKDISK* FVIRKDISK* RFINACCELPREM* DFINFRIKTION* SFVIRKDISK* SDOPSLAGOMK2DNLLAG*',
                 first, first, silent=1)

## 4. Solve the experiment

One step, with the options of the baseline run (ModelFlow remembers them). The Newton steps use the
factorization of the baseline. `newton_stall_refresh=1` is a safety net: if a step on it makes the largest
residual grow, the solve goes back to the point before the step and recalculates the derivatives there (one
factorization, about a minute) instead of diverging. A shock this small does not need it; public investment +1
per cent of GDP does (`makro26_experiment.ipynb`).

In [7]:
%%time
res = mmakro(alt, first, last, keep='Public purchases +1%', newton_stall_refresh=1)

CPU times: total: 2min 1s
Wall time: 13.2 s


The outcome of the solve is stored on the model: converged or not, the number of Newton steps, the largest
equation residual, and how often the derivatives were recalculated.

In [8]:
mmakro.ng_solve_status

{'solver': 'NewtonStackSolver',
 'converged': True,
 'iterations': 9,
 'reason': 'converged',
 'max_residual': 1.2341843103058636e-09,
 'refreshes': 0,
 'linesearch_steps': 0}

## 5. The impact

`mmakro[...]` compares the latest solution with the base. `#Headline` is a variable group stored with the
model. Quantities (`Q...`) and prices (`P...`) are growth- and inflation-adjusted in MAKRO; a percentage
difference from the baseline is the same in adjusted and ordinary units.

In [9]:
mmakro['#Headline']

## Or results in levels 

In [10]:
mmakro['#Headline'].rplot(datatype='level') 

## And the impact of the experiment in percent of the level
Other options are will show growth rates, impact on growth rate and so on 

In [11]:
mmakro['#Headline'].rplot(datatype='difpctlevel') 

## In a live jupyter notebook a interactive widget can be used 

In [12]:
mmakro.keep_plot_widget() 

## 6. What the model instance holds

`tools\model_info.py` reads only what is stored on the instance (counts, data, kept solutions, the
Jacobian the last solve left), for the time frame `mmakro.current_per` - after the solve the whole
simulation period 2030-2129. 

In [13]:
import sys
sys.path.insert(0, r'C:\mfmodeller_raw\makro\makro26\tools')
from model_info import model_info, jacobian_info

J1, JS = model_info(mmakro)

variables                                39,034
equations (endogenous)                   20,474
  normalized  X = f(...)                 20,100
  implicit    X___RES = f(...)           374
exogenous                                18,560
  solved through an implicit equation    374
  _D/_X fix switches                     3,304
  other inputs                           14,882
max lag / max lead                       -4 / 2
loaded straight (no structure analysis)  yes
time frame (current_per)                 2030-2129 (100)
lastdf                                   117 x 39,034 (2015-2131, 37 MB)
basedf                                   117 x 39,034 (2015-2131, 37 MB)
kept solutions                           Baseline, Public purchases +1%
stacked solve split pro / core / epi     2,834 / 15,472 / 2,168
stored derivative expressions            1 set(s)
last solver                              newtonstack_ng

time frame (current_per)                                   2030-2129 (100)
stacked J